In [10]:
import numpy as np
import pandas as pd
import pymongo
import sklearn
import matplotlib
import seaborn as sns

print('All required libraries are installed!')
print('pandas:', pd.__version__)
print('NumPy:', np.__version__)
print('PyMongo:', pymongo.version)
print('scikit-learn:', sklearn.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Seaborn:', sns.__version__)

All required libraries are installed!
pandas: 2.2.3
NumPy: 2.2.4
PyMongo: 4.19.0
scikit-learn: 1.6.1
Matplotlib: 3.10.1
Seaborn: 0.13.2


In [11]:

# CELL 1: LOAD AND VALIDATE THE DATASET

import json
import math
from pathlib import Path
from pprint import pprint
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

# Locate the project directory
DATASET_NAME = "bugradar issues og.json"
project_dir = Path.cwd()

while not (project_dir / "data" / DATASET_NAME).is_file():
    if project_dir.parent == project_dir:
        raise FileNotFoundError(
            f"Cannot find data/{DATASET_NAME}. "
            "Open the BugRadar project folder in VS Code."
        )
    project_dir = project_dir.parent

source_path = project_dir / "data" / DATASET_NAME

# Load the original JSON without modifying it
with source_path.open("r", encoding="utf-8") as file:
    original_data = json.load(file)

# Validate the MongoDB-style document structure
if not isinstance(original_data, list) or not original_data:
    raise ValueError("Expected a non-empty list of documents.")

if not all(isinstance(doc, dict) for doc in original_data):
    raise ValueError("Every record must be a JSON document/object.")

# Get all top-level fields across the documents
all_top_level_fields = sorted({
    key for doc in original_data for key in doc
})

print("Dataset loaded successfully!")
print("Total documents:", len(original_data))
print("Distinct top-level fields:", len(all_top_level_fields))
print("Dataset path:", source_path)


Dataset loaded successfully!
Total documents: 5000
Distinct top-level fields: 20
Dataset path: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar issues og.json


In [12]:

# CELL 2: INSPECT DOCUMENT STRUCTURE

# Display one complete document with its nested fields
print("SAMPLE MONGODB-STYLE DOCUMENT")
pprint(original_data[0], sort_dicts=False, width=100)

# Inspect nested documents and arrays
nested_fields = [
    "user", "repository", "environment",
    "milestone", "labels", "assignees"
]

structure = []

for field in nested_fields:
    present_values = [
        doc[field] for doc in original_data if field in doc
    ]

    structure.append({
        "Field": field,
        "Present": len(present_values),
        "Absent": len(original_data) - len(present_values),
        "Value types": dict(
            Counter(type(value).__name__ for value in present_values)
        )
    })

display(pd.DataFrame(structure).set_index("Field"))

print("\nTop-level fields:")
for field in all_top_level_fields:
    print("-", field)


SAMPLE MONGODB-STYLE DOCUMENT
{'id': 1000000,
 'number': 1,
 'title': 'Example missing for refresh the page',
 'body': None,
 'issue_category': 'documentation',
 'state': 'open',
 'state_reason': None,
 'priority': 'medium',
 'created_at': '2025-08-05T17:32:00Z',
 'closed_at': None,
 'resolution_hours': None,
 'labels': [],
 'comments': 7,
 'user': {'login': None, 'type': None},
 'assignees': [],
 'repository': {'name': 'auth-api', 'language': 'TypeScript', 'type': 'backend', 'stars': 144},
 'environment': {'platform': 'Firefox 131', 'version': '1.8.0'},
 'milestone': {'title': 'v2.0'},
 'url': 'https://example.invalid/auth-api/issues/1',
 'source': 'synthetic_educational_dataset'}


,Present,Absent,Value types
Field,,,
user,5000,0,"{'dict': 4713, 'NoneType': 287}"
repository,5000,0,{'dict': 5000}
environment,5000,0,"{'dict': 4385, 'NoneType': 615}"
milestone,5000,0,"{'dict': 1237, 'NoneType': 3763}"
labels,5000,0,{'list': 5000}
assignees,5000,0,{'list': 5000}



Top-level fields:
- assignees
- body
- closed_at
- comments
- created_at
- environment
- id
- issue_category
- labels
- milestone
- number
- priority
- repository
- resolution_hours
- source
- state
- state_reason
- title
- url
- user


In [13]:

# CELL 3: ANALYZE MISSING VALUES

total_docs = len(original_data)
missing_rows = []

# Check top-level fields directly in the original documents
for field in all_top_level_fields:
    values = [doc[field] for doc in original_data if field in doc]

    absent = sum(field not in doc for doc in original_data)
    nulls = sum(value is None for value in values)
    empty_strings = sum(
        isinstance(value, str) and value == "" for value in values
    )
    whitespace = sum(
        isinstance(value, str) and value != "" and not value.strip()
        for value in values
    )
    empty_arrays = sum(
        isinstance(value, list) and len(value) == 0 for value in values
    )

    missing_rows.append({
        "Field": field,
        "Absent": absent,
        "Null": nulls,
        "Empty strings": empty_strings,
        "Whitespace strings": whitespace,
        "Empty arrays": empty_arrays,
        "Missing/null %": round(
            (absent + nulls) / total_docs * 100, 2
        )
    })

missing_df = pd.DataFrame(missing_rows).set_index("Field")

print("TOP-LEVEL FIELD COMPLETENESS")
display(missing_df.sort_values("Missing/null %", ascending=False))

# Check embedded document availability
print("\nEMBEDDED DOCUMENT AVAILABILITY")

for field in ["user", "repository", "environment", "milestone"]:
    null_count = sum(doc.get(field) is None for doc in original_data)
    absent_count = sum(field not in doc for doc in original_data)
    present_count = sum(isinstance(doc.get(field), dict)
                        for doc in original_data)

    print(
        f"{field}: {present_count} objects, "
        f"{null_count} null, {absent_count} absent"
    )

# Check selected nested child fields
print("\nNESTED FIELD COMPLETENESS")

child_paths = [
    ("user", "login"),
    ("user", "type"),
    ("repository", "name"),
    ("repository", "language"),
    ("repository", "stars"),
    ("environment", "platform"),
    ("environment", "version"),
    ("milestone", "title")
]

child_rows = []

for parent, child in child_paths:
    absent_parent = null_parent = absent_child = null_child = 0

    for doc in original_data:
        if parent not in doc:
            absent_parent += 1
        elif doc[parent] is None:
            null_parent += 1
        elif isinstance(doc[parent], dict):
            if child not in doc[parent]:
                absent_child += 1
            elif doc[parent][child] is None:
                null_child += 1

    child_rows.append({
        "Nested field": f"{parent}.{child}",
        "Parent absent": absent_parent,
        "Parent null": null_parent,
        "Child absent": absent_child,
        "Child null": null_child
    })

display(pd.DataFrame(child_rows).set_index("Nested field"))


TOP-LEVEL FIELD COMPLETENESS


,Absent,Null,Empty strings,Whitespace strings,Empty arrays,Missing/null %
Field,,,,,,
milestone,0,3763,0,0,0,75.26
state_reason,0,2055,0,0,0,41.10
closed_at,0,1099,0,0,0,21.98
resolution_hours,0,1099,0,0,0,21.98
environment,0,615,0,0,0,12.30
body,0,372,0,0,0,7.44
user,0,287,0,0,0,5.74
comments,0,127,0,0,0,2.54
priority,0,60,72,0,0,1.20



EMBEDDED DOCUMENT AVAILABILITY
user: 4713 objects, 287 null, 0 absent
repository: 5000 objects, 0 null, 0 absent
environment: 4385 objects, 615 null, 0 absent
milestone: 1237 objects, 3763 null, 0 absent

NESTED FIELD COMPLETENESS


,Parent absent,Parent null,Child absent,Child null
Nested field,,,,
user.login,0,287,0,643
user.type,0,287,0,1504
repository.name,0,0,0,0
repository.language,0,0,0,782
repository.stars,0,0,0,0
environment.platform,0,615,0,417
environment.version,0,615,0,493
milestone.title,0,3763,0,0


In [14]:

# CELL 4: DETECT DUPLICATES

# Convert nested values to stable JSON strings for comparison
def stable_json(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False
    )

# Check duplicate application-level IDs and issue numbers
for field in ["id", "number"]:
    values = [
        stable_json(doc[field])
        for doc in original_data
        if field in doc and doc[field] is not None
    ]

    counts = Counter(values)
    duplicate_groups = {
        value: count for value, count in counts.items()
        if count > 1
    }

    extra_records = sum(
        count - 1 for count in duplicate_groups.values()
    )

    print(f"Duplicate {field} groups:", len(duplicate_groups))
    print(f"Extra records with repeated {field}:", extra_records)

# Check exact duplicate documents
document_strings = [
    stable_json(doc) for doc in original_data
]

document_counts = Counter(document_strings)
exact_duplicates = {
    doc: count for doc, count in document_counts.items()
    if count > 1
}

print("\nExact duplicate document groups:", len(exact_duplicates))
print(
    "Extra exact duplicate documents:",
    sum(count - 1 for count in exact_duplicates.values())
)

# Check repeated labels inside each document
repeated_label_docs = 0
repeated_label_entries = 0

for doc in original_data:
    labels = doc.get("labels")

    if not isinstance(labels, list):
        continue

    label_names = [
        stable_json(label.get("name", label))
        if isinstance(label, dict) else stable_json(label)
        for label in labels
    ]

    extra = len(label_names) - len(set(label_names))

    if extra > 0:
        repeated_label_docs += 1
        repeated_label_entries += extra

print("Documents with repeated labels:", repeated_label_docs)
print("Extra repeated label entries:", repeated_label_entries)


Duplicate id groups: 0
Extra records with repeated id: 0
Duplicate number groups: 0
Extra records with repeated number: 0

Exact duplicate document groups: 0
Extra exact duplicate documents: 0
Documents with repeated labels: 1715
Extra repeated label entries: 1913


In [15]:

# CELL 5: CHECK INCONSISTENT FORMATS

# Inspect distinct values in important categorical fields
for field in ["issue_category", "priority", "state"]:
    print(f"\nVALUES IN {field.upper()}")

    counts = Counter(
        stable_json(doc[field])
        for doc in original_data
        if field in doc
    )

    display(pd.DataFrame([
        {"Value": value, "Count": count}
        for value, count in counts.items()
    ]).sort_values("Count", ascending=False))

# Count capitalization and whitespace inconsistencies
print("\nSTRING CONSISTENCY CHECK")

for field in ["issue_category", "priority", "state"]:
    values = [
        doc[field] for doc in original_data
        if isinstance(doc.get(field), str)
    ]

    normalized_groups = {}

    for value in values:
        normalized_groups.setdefault(
            value.strip().casefold(), set()
        ).add(value)

    variants = {
        key: sorted(items)
        for key, items in normalized_groups.items()
        if len(items) > 1
    }

    whitespace_count = sum(
        value != value.strip() for value in values
    )

    print(f"\n{field}:")
    print("Values with leading/trailing whitespace:", whitespace_count)
    print("Case/whitespace variant groups:", variants)

# Check missing or blank text
print("\nTITLE AND BODY COMPLETENESS")

for field in ["title", "body"]:
    absent = sum(field not in doc for doc in original_data)
    nulls = sum(doc.get(field) is None for doc in original_data)
    blank = sum(
        isinstance(doc.get(field), str)
        and not doc[field].strip()
        for doc in original_data
    )

    print(
        f"{field}: absent={absent}, null={nulls}, blank={blank}"
    )

# Check data types of key scalar fields
print("\nSCALAR FIELD DATA TYPES")

for field in ["id", "comments", "resolution_hours"]:
    types = Counter(
        type(doc[field]).__name__
        for doc in original_data
        if field in doc and doc[field] is not None
    )

    print(f"{field}: {dict(types)}")



VALUES IN ISSUE_CATEGORY


,Value,Count
3,"""bug""",2749
2,"""enhancement""",937
0,"""documentation""",640
4,"""question""",443
6,"""Bug""",76
7,"""BUG""",56
1,"""Enhancement""",29
9,"""ENHANCEMENT""",24
5,"""Documentation""",16
8,"""QUESTION""",15



VALUES IN PRIORITY


,Value,Count
0,"""medium""",2466
1,"""low""",1192
3,"""high""",833
2,"""critical""",254
7,"""""",72
5,"""HIGH""",71
6,null,60
4,"""Medium""",52



VALUES IN STATE


,Value,Count
1,"""closed""",3901
0,"""open""",1099



STRING CONSISTENCY CHECK

issue_category:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {'documentation': ['DOCUMENTATION', 'Documentation', 'documentation'], 'enhancement': ['ENHANCEMENT', 'Enhancement', 'enhancement'], 'bug': ['BUG', 'Bug', 'bug'], 'question': ['QUESTION', 'Question', 'question']}

priority:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {'medium': ['Medium', 'medium'], 'high': ['HIGH', 'high']}

state:
Values with leading/trailing whitespace: 0
Case/whitespace variant groups: {}

TITLE AND BODY COMPLETENESS
title: absent=0, null=0, blank=0
body: absent=0, null=372, blank=0

SCALAR FIELD DATA TYPES
id: {'int': 5000}
comments: {'int': 4743, 'str': 130}
resolution_hours: {'float': 3896, 'int': 5}


In [16]:

# CELL 6: VALIDATE TIMESTAMPS AND RESOLUTION TIME

def parse_timestamp(value):
    """Parse a timestamp for validation without changing the source."""
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        parsed = datetime.fromisoformat(
            value.strip().replace("Z", "+00:00")
        )

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        return parsed

    except (ValueError, TypeError, OverflowError):
        return None


timestamp_counts = Counter()
closed_before_created = 0
open_with_close = 0
closed_without_close = 0
missing_resolution = 0
invalid_resolution = 0
negative_resolution = 0
duration_mismatches = 0
comparable_durations = 0

for doc in original_data:
    created_raw = doc.get("created_at")
    closed_raw = doc.get("closed_at")

    created = parse_timestamp(created_raw)
    closed = parse_timestamp(closed_raw)

    # Record timestamp issues
    for field, raw, parsed in [
        ("created_at", created_raw, created),
        ("closed_at", closed_raw, closed)
    ]:
        if field not in doc:
            timestamp_counts[f"{field}: absent"] += 1
        elif raw is None:
            timestamp_counts[f"{field}: null"] += 1
        elif parsed is None:
            timestamp_counts[f"{field}: invalid"] += 1

    state = doc.get("state")
    state = state.strip().casefold() if isinstance(state, str) else None

    has_close_value = (
        closed_raw is not None
        and not (
            isinstance(closed_raw, str) and not closed_raw.strip()
        )
    )

    if created is not None and closed is not None:
        if closed < created:
            closed_before_created += 1

    if state == "open" and has_close_value:
        open_with_close += 1

    if state == "closed" and not has_close_value:
        closed_without_close += 1

    # Validate stored resolution_hours
    resolution = doc.get("resolution_hours")

    if resolution is None:
        missing_resolution += 1
        continue

    if (
        isinstance(resolution, bool)
        or not isinstance(resolution, (int, float))
        or not math.isfinite(resolution)
    ):
        invalid_resolution += 1
        continue

    if resolution < 0:
        negative_resolution += 1
        continue

    # Compare with elapsed time when both timestamps are valid
    if created is not None and closed is not None and closed >= created:
        calculated = (closed - created).total_seconds() / 3600
        comparable_durations += 1

        # Stored values are rounded to 2 decimal places
        if abs(float(resolution) - calculated) > 0.005001:
            duration_mismatches += 1

print("TIMESTAMP VALIDATION")
for issue, count in timestamp_counts.items():
    print(f"{issue}: {count}")

print("\nRESOLUTION VALIDATION")
print("Closing date before creation date:", closed_before_created)
print("Open issues with closing timestamp:", open_with_close)
print("Closed issues without closing timestamp:", closed_without_close)
print("Missing resolution_hours:", missing_resolution)
print("Invalid resolution_hours:", invalid_resolution)
print("Negative resolution_hours:", negative_resolution)
print("Comparable durations:", comparable_durations)
print("Duration mismatches beyond rounding tolerance:", duration_mismatches)


TIMESTAMP VALIDATION
closed_at: null: 1099
closed_at: invalid: 22
created_at: null: 30

RESOLUTION VALIDATION
Closing date before creation date: 0
Open issues with closing timestamp: 0
Closed issues without closing timestamp: 0
Missing resolution_hours: 1099
Invalid resolution_hours: 0
Negative resolution_hours: 0
Comparable durations: 3857
Duration mismatches beyond rounding tolerance: 0


In [18]:

# CELL 7: DATA-QUALITY SUMMARY

print("=" * 50)
print("BUGRADAR — DATASET EXPLORATION SUMMARY")
print("=" * 50)

print("Total documents:", len(original_data))
print("Top-level fields:", len(all_top_level_fields))

print(
    "Documents with repeated labels:",
    repeated_label_docs
)

print(
    "Exact duplicate document groups:",
    len(exact_duplicates)
)

print(
    "Issues closed before creation:",
    closed_before_created
)

print(
    "Open issues with closing timestamps:",
    open_with_close
)

print(
    "Closed issues without closing timestamps:",
    closed_without_close
)

print(
    "Resolution duration mismatches:",
    duration_mismatches
)



BUGRADAR — DATASET EXPLORATION SUMMARY
Total documents: 5000
Top-level fields: 20
Documents with repeated labels: 1715
Exact duplicate document groups: 0
Issues closed before creation: 0
Open issues with closing timestamps: 0
Closed issues without closing timestamps: 0
Resolution duration mismatches: 0



## 3. Data Cleaning and Preprocessing

In this section, the original MongoDB-style issue documents
are cleaned while preserving embedded documents and arrays.

The preprocessing includes:
- Removing duplicate issue documents using the issue ID.
- Standardizing text fields and categorical values.
- Handling missing and invalid values.
- Normalizing labels and validating comments.
- Validating timestamps and resolution durations.
- Saving the cleaned documents as a separate JSON file.

The original dataset remains unchanged.


### 3.1 Prepare a Safe Copy and Review Missing Values

The next cell checks that the already-loaded `original_data` matches the UTF-8 JSON source, then creates independent deep copies. Missing keys and explicit `null` values remain distinct. Blank text becomes `null`; other missing values are preserved. Missing embedded documents are never fabricated, and model-specific imputation is deferred until feature requirements are known.

In [26]:

# 3.1: Import dependencies and validate the source dataset

import copy as preprocessing_copy
import json as preprocessing_json
import math as preprocessing_math
from collections import Counter as PreprocessingCounter
from datetime import datetime as PreprocessingDateTime, timezone as PreprocessingTimezone
from pathlib import Path as PreprocessingPath
from pprint import pprint as preprocessing_pprint

import pandas as preprocessing_pd
from IPython.display import display as preprocessing_display

# Confirm that the existing dataset-loading section has been executed.
if 'original_data' not in globals():
    raise NameError(
        'Run the existing dataset-loading section first; '
        'original_data is not defined.'
    )

if not isinstance(original_data, list) or not original_data:
    raise ValueError(
        'original_data must be a non-empty list of issue-document dictionaries.'
    )

if not all(isinstance(preprocessing_document, dict)
           for preprocessing_document in original_data):
    raise ValueError(
        'Every item in original_data must be a dictionary/document.'
    )

# Locate the original JSON file without hardcoding the absolute path.
preprocessing_dataset_name = 'bugradar issues og.json'

preprocessing_project_dir = next(
    (
        preprocessing_folder
        for preprocessing_folder in (
            PreprocessingPath.cwd(),
            *PreprocessingPath.cwd().parents
        )
        if (
            preprocessing_folder / 'data' /
            preprocessing_dataset_name
        ).is_file()
    ),
    None,
)

if preprocessing_project_dir is None:
    raise FileNotFoundError(
        f'Could not locate data/{preprocessing_dataset_name} '
        f'from {PreprocessingPath.cwd()}.'
    )

preprocessing_source_path = (
    preprocessing_project_dir / 'data' / preprocessing_dataset_name
)

# Load and validate the source JSON.
try:
    with preprocessing_source_path.open('r', encoding='utf-8') as preprocessing_file:
        preprocessing_source_data = preprocessing_json.load(preprocessing_file)

except (
    OSError,
    UnicodeDecodeError,
    preprocessing_json.JSONDecodeError
) as preprocessing_error:
    raise ValueError(
        f'Could not read the original JSON file: {preprocessing_error}'
    ) from preprocessing_error

if (
    not isinstance(preprocessing_source_data, list)
    or not all(
        isinstance(preprocessing_document, dict)
        for preprocessing_document in preprocessing_source_data
    )
):
    raise ValueError(
        'The source JSON must be a list of issue-document dictionaries.'
    )

if original_data != preprocessing_source_data:
    raise RuntimeError(
        'original_data does not match the source JSON; '
        'reload the exploration section before preprocessing.'
    )

print("Source dataset validated successfully.")
print("Source file:", preprocessing_source_path)


Source dataset validated successfully.
Source file: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar issues og.json


In [27]:

# 3.2: Create a snapshot and a separate working copy

# Preserve a snapshot for checking that the original data remains unchanged.
preprocessing_original_snapshot = preprocessing_copy.deepcopy(original_data)

# All preprocessing operations will use this independent working copy.
preprocessing_data = preprocessing_copy.deepcopy(original_data)

preprocessing_original_count = len(preprocessing_original_snapshot)

print(
    f'Confirmed {preprocessing_original_count:,} loaded documents '
    f'match {preprocessing_source_path}.'
)

print(
    'Created independent deep copies; original_data and '
    'the source JSON will not be edited.'
)


Confirmed 5,000 loaded documents match c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar issues og.json.
Created independent deep copies; original_data and the source JSON will not be edited.


In [28]:

# 3.3: Audit missing values in the original top-level fields

preprocessing_top_fields = sorted({
    preprocessing_key
    for preprocessing_document in preprocessing_original_snapshot
    for preprocessing_key in preprocessing_document
})

preprocessing_missing_rows = []

for preprocessing_field in preprocessing_top_fields:

    # Count fields absent from a document.
    preprocessing_absent = sum(
        preprocessing_field not in preprocessing_document
        for preprocessing_document in preprocessing_original_snapshot
    )

    preprocessing_values = [
        preprocessing_document[preprocessing_field]
        for preprocessing_document in preprocessing_original_snapshot
        if preprocessing_field in preprocessing_document
    ]

    # Count different representations of missing or empty values.
    preprocessing_null = sum(
        preprocessing_value is None
        for preprocessing_value in preprocessing_values
    )

    preprocessing_empty = sum(
        isinstance(preprocessing_value, str)
        and preprocessing_value == ''
        for preprocessing_value in preprocessing_values
    )

    preprocessing_whitespace = sum(
        isinstance(preprocessing_value, str)
        and preprocessing_value != ''
        and not preprocessing_value.strip()
        for preprocessing_value in preprocessing_values
    )

    preprocessing_empty_arrays = sum(
        isinstance(preprocessing_value, list)
        and not preprocessing_value
        for preprocessing_value in preprocessing_values
    )

    preprocessing_missing_rows.append({
        'field': preprocessing_field,
        'absent': preprocessing_absent,
        'null': preprocessing_null,
        'empty string': preprocessing_empty,
        'whitespace-only string': preprocessing_whitespace,
        'empty array': preprocessing_empty_arrays
    })

print('Original top-level missing-value audit:')
preprocessing_display(
    preprocessing_pd.DataFrame(preprocessing_missing_rows)
    .set_index('field')
)


Original top-level missing-value audit:


,absent,null,empty string,whitespace-only string,empty array
field,,,,,
assignees,0,0,0,0,3504
body,0,372,0,0,0
closed_at,0,1099,0,0,0
comments,0,127,0,0,0
created_at,0,30,0,0,0
environment,0,615,0,0,0
id,0,0,0,0,0
issue_category,0,0,0,0,0
labels,0,0,0,0,386


In [29]:

# 3.4: Audit nested MongoDB-style documents and their child fields

# Check whether embedded documents are absent, null, or the wrong type.
preprocessing_nested_rows = []

for preprocessing_parent in (
    'user', 'repository', 'environment', 'milestone'
):

    preprocessing_absent = 0
    preprocessing_null = 0
    preprocessing_wrong_type = 0

    for preprocessing_document in preprocessing_original_snapshot:

        if preprocessing_parent not in preprocessing_document:
            preprocessing_absent += 1

        elif preprocessing_document[preprocessing_parent] is None:
            preprocessing_null += 1

        elif not isinstance(
            preprocessing_document[preprocessing_parent], dict
        ):
            preprocessing_wrong_type += 1

    preprocessing_nested_rows.append({
        'embedded field': preprocessing_parent,
        'absent': preprocessing_absent,
        'null': preprocessing_null,
        'unexpected type': preprocessing_wrong_type
    })

print('Original embedded-document availability:')

preprocessing_display(
    preprocessing_pd.DataFrame(preprocessing_nested_rows)
    .set_index('embedded field')
)

# Check selected child fields while accounting for missing parent documents.
preprocessing_child_rows = []

for preprocessing_parent, preprocessing_child in (
    ('user', 'login'),
    ('user', 'type'),
    ('repository', 'language'),
    ('environment', 'platform'),
    ('milestone', 'title')
):

    preprocessing_counts = PreprocessingCounter()

    for preprocessing_document in preprocessing_original_snapshot:

        if preprocessing_parent not in preprocessing_document:
            preprocessing_counts['parent absent'] += 1

        elif preprocessing_document[preprocessing_parent] is None:
            preprocessing_counts['parent null'] += 1

        elif not isinstance(
            preprocessing_document[preprocessing_parent], dict
        ):
            preprocessing_counts['parent wrong type'] += 1

        elif preprocessing_child not in preprocessing_document[
            preprocessing_parent
        ]:
            preprocessing_counts['child absent'] += 1

        else:
            preprocessing_value = preprocessing_document[
                preprocessing_parent
            ][preprocessing_child]

            if preprocessing_value is None:
                preprocessing_counts['child null'] += 1

            elif (
                isinstance(preprocessing_value, str)
                and not preprocessing_value.strip()
            ):
                preprocessing_counts['child blank'] += 1

    preprocessing_child_rows.append({
        'path': f'{preprocessing_parent}.{preprocessing_child}',
        **preprocessing_counts
    })

print('Original nested-child missing-value audit:')

preprocessing_display(
    preprocessing_pd.DataFrame(preprocessing_child_rows)
    .fillna(0)
    .set_index('path')
)


Original embedded-document availability:


,absent,null,unexpected type
embedded field,,,
user,0,287,0
repository,0,0,0
environment,0,615,0
milestone,0,3763,0


Original nested-child missing-value audit:


,child null,child blank,parent null
path,,,
user.login,643.0,689.0,287.0
user.type,1504.0,0.0,287.0
repository.language,782.0,0.0,0.0
environment.platform,417.0,0.0,615.0
milestone.title,0.0,0.0,3763.0


### 3.2 Remove Duplicate IDs and Standardize Text

Duplicate detection keeps the first record for each non-missing `id`. Missing IDs are retained and never grouped together. Categorical strings are trimmed and case-folded without filtering unknown values. Nested objects and arrays are cleaned only when their actual Python types match the expected structure.

In [30]:

# 3.5: Handle duplicate issue IDs while retaining records with missing IDs

preprocessing_missing_id_count = 0
preprocessing_seen_ids = set()
preprocessing_unique_data = []
preprocessing_duplicate_records_removed = 0

for preprocessing_document in preprocessing_data:
    preprocessing_issue_id = preprocessing_document.get('id')

    preprocessing_id_missing = (
        'id' not in preprocessing_document
        or preprocessing_issue_id is None
        or (
            isinstance(preprocessing_issue_id, str)
            and not preprocessing_issue_id.strip()
        )
    )

    if preprocessing_id_missing:
        preprocessing_missing_id_count += 1
        preprocessing_unique_data.append(preprocessing_document)
        continue

    preprocessing_id_key = (
        type(preprocessing_issue_id).__name__,
        preprocessing_json.dumps(
            preprocessing_issue_id,
            sort_keys=True,
            ensure_ascii=False
        )
    )

    if preprocessing_id_key in preprocessing_seen_ids:
        preprocessing_duplicate_records_removed += 1
        continue

    preprocessing_seen_ids.add(preprocessing_id_key)
    preprocessing_unique_data.append(preprocessing_document)

preprocessing_data = preprocessing_unique_data

print(f'Missing IDs retained: {preprocessing_missing_id_count}')
print(f'Duplicate records removed by id: {preprocessing_duplicate_records_removed}')
print(f'Records remaining after duplicate handling: {len(preprocessing_data):,}')


Missing IDs retained: 0
Duplicate records removed by id: 0
Records remaining after duplicate handling: 5,000


In [31]:

# 3.6: Validate comment counts and convert valid numeric strings

preprocessing_invalid_comments_count = 0
preprocessing_comment_strings_converted = 0

for preprocessing_document in preprocessing_data:
    preprocessing_comments = preprocessing_document.get('comments')

    if 'comments' not in preprocessing_document or preprocessing_comments is None:
        continue

    if isinstance(preprocessing_comments, str):
        try:
            preprocessing_parsed_comments = int(preprocessing_comments.strip())
        except ValueError:
            preprocessing_parsed_comments = None

        if (
            preprocessing_parsed_comments is not None
            and preprocessing_parsed_comments >= 0
        ):
            preprocessing_document['comments'] = preprocessing_parsed_comments
            preprocessing_comment_strings_converted += 1
            continue

    elif (
        isinstance(preprocessing_comments, int)
        and not isinstance(preprocessing_comments, bool)
        and preprocessing_comments >= 0
    ):
        continue

    preprocessing_invalid_comments_count += 1
    preprocessing_document['comments'] = None

print(f'Invalid comment counts set to null: {preprocessing_invalid_comments_count}')
print(f'Numeric comment strings converted to integers: {preprocessing_comment_strings_converted}')


Invalid comment counts set to null: 0
Numeric comment strings converted to integers: 130


In [32]:

# 3.7: Trim whitespace and standardize categorical values

preprocessing_categorical_fields = {
    'issue_category',
    'priority',
    'state'
}

preprocessing_categories_before = {
    preprocessing_field: PreprocessingCounter(
        repr(preprocessing_document.get(preprocessing_field))
        for preprocessing_document in preprocessing_data
    )
    for preprocessing_field in preprocessing_categorical_fields
}

for preprocessing_document in preprocessing_data:
    for preprocessing_field, preprocessing_value in list(preprocessing_document.items()):

        if (
            not isinstance(preprocessing_value, str)
            or preprocessing_field == 'id'
        ):
            continue

        if preprocessing_field in preprocessing_categorical_fields:
            preprocessing_clean_value = preprocessing_value.strip().casefold()

        elif preprocessing_field not in {'created_at', 'closed_at'}:
            preprocessing_clean_value = preprocessing_value.strip()

        else:
            continue

        preprocessing_document[preprocessing_field] = (
            preprocessing_clean_value or None
        )

# Clean recognized string fields inside embedded documents.
preprocessing_nested_text = {
    'user': ('login', 'type'),
    'repository': ('name', 'language', 'type'),
    'environment': ('platform', 'version'),
    'milestone': ('title',)
}

for preprocessing_document in preprocessing_data:
    for preprocessing_parent, preprocessing_children in preprocessing_nested_text.items():

        preprocessing_embedded = preprocessing_document.get(preprocessing_parent)

        if isinstance(preprocessing_embedded, dict):
            for preprocessing_child in preprocessing_children:
                preprocessing_value = preprocessing_embedded.get(preprocessing_child)

                if isinstance(preprocessing_value, str):
                    preprocessing_embedded[preprocessing_child] = (
                        preprocessing_value.strip() or None
                    )

print('Top-level and embedded text standardization completed.')


Top-level and embedded text standardization completed.


In [33]:

# 3.8: Standardize strings in nested arrays without flattening them

for preprocessing_document in preprocessing_data:
    for preprocessing_array, preprocessing_child in (
        ('labels', 'name'),
        ('assignees', 'login')
    ):
        preprocessing_values = preprocessing_document.get(preprocessing_array)

        if isinstance(preprocessing_values, list):
            for preprocessing_item_index, preprocessing_item in enumerate(preprocessing_values):

                if isinstance(preprocessing_item, dict):
                    preprocessing_value = preprocessing_item.get(preprocessing_child)

                    if isinstance(preprocessing_value, str):
                        preprocessing_item[preprocessing_child] = (
                            preprocessing_value.strip() or None
                        )

                elif isinstance(preprocessing_item, str):
                    preprocessing_values[preprocessing_item_index] = (
                        preprocessing_item.strip() or None
                    )

# Count category values after standardization.
preprocessing_categories_after = {}

for preprocessing_field in sorted(preprocessing_categorical_fields):
    preprocessing_categories_after[preprocessing_field] = dict(
        PreprocessingCounter(
            repr(preprocessing_document.get(preprocessing_field))
            for preprocessing_document in preprocessing_data
        )
    )

print('Categorical values before standardization:')
preprocessing_pprint({
    preprocessing_field: dict(preprocessing_counts)
    for preprocessing_field, preprocessing_counts
    in preprocessing_categories_before.items()
})

print('\nCategorical values after standardization:')
preprocessing_pprint(preprocessing_categories_after)

print('\nNested arrays and embedded dictionaries remain intact.')


Categorical values before standardization:
{'issue_category': {"'BUG'": 56,
                    "'Bug'": 76,
                    "'DOCUMENTATION'": 9,
                    "'Documentation'": 16,
                    "'ENHANCEMENT'": 24,
                    "'Enhancement'": 29,
                    "'QUESTION'": 15,
                    "'Question'": 6,
                    "'bug'": 2749,
                    "'documentation'": 640,
                    "'enhancement'": 937,
                    "'question'": 443},
 'priority': {"''": 72,
              "'HIGH'": 71,
              "'Medium'": 52,
              "'critical'": 254,
              "'high'": 833,
              "'low'": 1192,
              "'medium'": 2466,
              'None': 60},
 'state': {"'closed'": 3901, "'open'": 1099}}

Categorical values after standardization:
{'issue_category': {"'bug'": 2881,
                    "'documentation'": 665,
                    "'enhancement'": 990,
                    "'question'": 464},
 'prio

### 3.3 Validate Timestamps and Resolution Durations

Timestamp parsing is temporary. Malformed non-empty timestamps are set to `null` in the cleaned copy, while missing values remain missing. A duration is recalculated only when both timestamps parse and the close time is not earlier than the creation time. Recalculated hours are rounded to two decimal places; for inconsistent dates, existing duration and state values are preserved and reported.

In [34]:

# 3.9: Define timestamp parsing and initialize validation counters

def preprocessing_parse_timestamp(preprocessing_value):
    """Parse ISO timestamps, returning None for missing or invalid values."""
    if not isinstance(preprocessing_value, str) or not preprocessing_value.strip():
        return None

    try:
        preprocessing_parsed = PreprocessingDateTime.fromisoformat(
            preprocessing_value.strip().replace('Z', '+00:00')
        )

        if preprocessing_parsed.tzinfo is None:
            preprocessing_parsed = preprocessing_parsed.replace(
                tzinfo=PreprocessingTimezone.utc
            )

        return preprocessing_parsed

    except ValueError:
        return None


# Initialize timestamp audit counters.
preprocessing_timestamp_counts = {
    preprocessing_field: PreprocessingCounter()
    for preprocessing_field in ('created_at', 'closed_at')
}

preprocessing_invalid_timestamp_records = set()
preprocessing_closed_before_created = 0
preprocessing_open_with_close = 0
preprocessing_closed_without_close = 0

# Initialize resolution-time validation counters.
preprocessing_invalid_resolution_count = 0
preprocessing_negative_resolution_count = 0
preprocessing_missing_resolution_count = 0
preprocessing_closed_missing_resolution_count = 0
preprocessing_resolution_mismatch_count = 0
preprocessing_duration_recalculated_count = 0

# Allow a small difference caused by rounding durations to two decimal places.
preprocessing_rounding_tolerance_hours = 0.005000001

print("Timestamp parser and validation counters initialized.")


Timestamp parser and validation counters initialized.


In [35]:

# 3.10: Parse timestamps and validate resolution times

for preprocessing_document_index, preprocessing_document in enumerate(preprocessing_data):

    preprocessing_raw_closed = preprocessing_document.get('closed_at')

    preprocessing_has_close_value = (
        'closed_at' in preprocessing_document
        and preprocessing_raw_closed is not None
        and not (
            isinstance(preprocessing_raw_closed, str)
            and not preprocessing_raw_closed.strip()
        )
    )

    preprocessing_parsed_dates = {}

    # Parse creation and closing timestamps.
    for preprocessing_field in ('created_at', 'closed_at'):

        if preprocessing_field not in preprocessing_document:
            preprocessing_timestamp_counts[preprocessing_field]['absent'] += 1
            preprocessing_parsed_dates[preprocessing_field] = None
            continue

        preprocessing_raw_timestamp = preprocessing_document[preprocessing_field]

        if (
            preprocessing_raw_timestamp is None
            or (
                isinstance(preprocessing_raw_timestamp, str)
                and not preprocessing_raw_timestamp.strip()
            )
        ):
            preprocessing_timestamp_counts[preprocessing_field][
                'missing/null/blank'
            ] += 1

            preprocessing_document[preprocessing_field] = None
            preprocessing_parsed_dates[preprocessing_field] = None
            continue

        preprocessing_parsed = preprocessing_parse_timestamp(
            preprocessing_raw_timestamp
        )

        if preprocessing_parsed is None:
            preprocessing_timestamp_counts[preprocessing_field]['invalid'] += 1
            preprocessing_invalid_timestamp_records.add(
                preprocessing_document_index
            )
            preprocessing_document[preprocessing_field] = None

        elif isinstance(preprocessing_raw_timestamp, str):
            preprocessing_document[preprocessing_field] = (
                preprocessing_raw_timestamp.strip()
            )

        preprocessing_parsed_dates[preprocessing_field] = preprocessing_parsed

    preprocessing_created = preprocessing_parsed_dates['created_at']
    preprocessing_closed = preprocessing_parsed_dates['closed_at']
    preprocessing_state = preprocessing_document.get('state')

    # Check consistency between issue state and closing timestamp.
    if preprocessing_state == 'open' and preprocessing_has_close_value:
        preprocessing_open_with_close += 1

    if preprocessing_state == 'closed' and not preprocessing_has_close_value:
        preprocessing_closed_without_close += 1

    # Check for closing dates that precede creation dates.
    if (
        preprocessing_created is not None
        and preprocessing_closed is not None
        and preprocessing_closed < preprocessing_created
    ):
        preprocessing_closed_before_created += 1

    # Validate the stored resolution duration.
    preprocessing_stored_duration = preprocessing_document.get(
        'resolution_hours'
    )

    if (
        'resolution_hours' not in preprocessing_document
        or preprocessing_stored_duration is None
    ):
        preprocessing_missing_resolution_count += 1

        if preprocessing_state == 'closed':
            preprocessing_closed_missing_resolution_count += 1

    elif (
        isinstance(preprocessing_stored_duration, bool)
        or not isinstance(preprocessing_stored_duration, (int, float))
        or not preprocessing_math.isfinite(preprocessing_stored_duration)
    ):
        preprocessing_invalid_resolution_count += 1
        preprocessing_document['resolution_hours'] = None

    elif preprocessing_stored_duration < 0:
        preprocessing_negative_resolution_count += 1
        preprocessing_document['resolution_hours'] = None

    # Recalculate only when both timestamps are valid and correctly ordered.
    if (
        preprocessing_created is not None
        and preprocessing_closed is not None
        and preprocessing_closed >= preprocessing_created
    ):
        preprocessing_calculated_hours = (
            preprocessing_closed - preprocessing_created
        ).total_seconds() / 3600

        if (
            isinstance(preprocessing_stored_duration, (int, float))
            and not isinstance(preprocessing_stored_duration, bool)
            and preprocessing_math.isfinite(preprocessing_stored_duration)
            and preprocessing_stored_duration >= 0
            and abs(
                preprocessing_stored_duration
                - preprocessing_calculated_hours
            ) > preprocessing_rounding_tolerance_hours
        ):
            preprocessing_resolution_mismatch_count += 1

        preprocessing_document['resolution_hours'] = round(
            preprocessing_calculated_hours, 2
        )

        preprocessing_duration_recalculated_count += 1

print("Date parsing and resolution-time validation completed.")


Date parsing and resolution-time validation completed.


In [36]:

# 3.11: Display timestamp and resolution-time validation results

print("Timestamp audit by field:")
preprocessing_display(
    preprocessing_pd.DataFrame.from_dict(
        preprocessing_timestamp_counts,
        orient='index'
    ).fillna(0)
)

print(
    "Records with at least one invalid timestamp:",
    len(preprocessing_invalid_timestamp_records)
)

print(
    "Closing timestamps earlier than creation timestamps:",
    preprocessing_closed_before_created
)

print(
    "Open issues with a closing timestamp:",
    preprocessing_open_with_close
)

print(
    "Closed issues without a closing timestamp:",
    preprocessing_closed_without_close
)

print(
    "Missing stored resolution values:",
    preprocessing_missing_resolution_count
)

print(
    "Closed issues missing a stored resolution value:",
    preprocessing_closed_missing_resolution_count
)

print(
    "Invalid/non-numeric/non-finite resolution values:",
    preprocessing_invalid_resolution_count
)

print(
    "Negative resolution values:",
    preprocessing_negative_resolution_count
)

print(
    "Stored durations inconsistent beyond rounding tolerance:",
    preprocessing_resolution_mismatch_count
)

print(
    "Durations recalculated from valid, ordered timestamp pairs:",
    preprocessing_duration_recalculated_count
)


Timestamp audit by field:


,missing/null/blank,invalid
created_at,30,0.0
closed_at,1099,22.0


Records with at least one invalid timestamp: 22
Closing timestamps earlier than creation timestamps: 0
Open issues with a closing timestamp: 0
Closed issues without a closing timestamp: 0
Missing stored resolution values: 1099
Closed issues missing a stored resolution value: 0
Invalid/non-numeric/non-finite resolution values: 0
Negative resolution values: 0
Stored durations inconsistent beyond rounding tolerance: 0
Durations recalculated from valid, ordered timestamp pairs: 3857


### 3.4 Save, Reload, and Validate the Cleaned Dataset

The cleaned copy is written as UTF-8 JSON to `data/bugradar_issues_cleaned.json`. The final cell reloads that file, checks its record count and nested structure, compares the in-memory source to its deep-copy snapshot, and prints the actual preprocessing summary and one representative cleaned document.

In [37]:

# 3.12: Save and reload the cleaned dataset

preprocessing_output_path = (
    preprocessing_project_dir / 'data' / 'bugradar_issues_cleaned.json'
)

# Ensure the data folder exists.
preprocessing_output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the cleaned documents as UTF-8 JSON.
try:
    with preprocessing_output_path.open('w', encoding='utf-8') as preprocessing_output_file:
        preprocessing_json.dump(
            preprocessing_data,
            preprocessing_output_file,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        )

except (OSError, TypeError, ValueError) as preprocessing_error:
    raise RuntimeError(
        f'Could not serialize the cleaned dataset to '
        f'{preprocessing_output_path}: {preprocessing_error}'
    ) from preprocessing_error

# Reload the saved file to verify that it is valid JSON.
with preprocessing_output_path.open('r', encoding='utf-8') as preprocessing_saved_file:
    preprocessing_cleaned_data = preprocessing_json.load(
        preprocessing_saved_file
    )

print("Cleaned dataset saved and reloaded.")
print("Output path:", preprocessing_output_path)
print("Reloaded records:", len(preprocessing_cleaned_data))


Cleaned dataset saved and reloaded.
Output path: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar_issues_cleaned.json
Reloaded records: 5000


In [38]:

# 3.13: Validate the saved dataset and summarize preprocessing

# Verify that the saved file contains a list of document dictionaries.
if (
    not isinstance(preprocessing_cleaned_data, list)
    or not all(
        isinstance(preprocessing_document, dict)
        for preprocessing_document in preprocessing_cleaned_data
    )
):
    raise ValueError(
        'The saved cleaned file is not a JSON list of document dictionaries.'
    )

# Verify record count and content.
if len(preprocessing_cleaned_data) != len(preprocessing_data):
    raise AssertionError(
        'Saved document count does not match the cleaned in-memory count.'
    )

if preprocessing_cleaned_data != preprocessing_data:
    raise AssertionError(
        'Reloaded JSON does not match the cleaned in-memory documents.'
    )

# Confirm embedded documents and arrays remain nested.
for preprocessing_document in preprocessing_cleaned_data:

    for preprocessing_field in (
        'user', 'repository', 'environment', 'milestone'
    ):
        if isinstance(preprocessing_document.get(preprocessing_field), dict):
            assert isinstance(
                preprocessing_document[preprocessing_field], dict
            )

    for preprocessing_field in ('labels', 'assignees'):
        if isinstance(preprocessing_document.get(preprocessing_field), list):
            assert isinstance(
                preprocessing_document[preprocessing_field], list
            )

            for preprocessing_item in preprocessing_document[preprocessing_field]:
                if isinstance(preprocessing_item, dict):
                    assert isinstance(preprocessing_item, dict)

# Verify that the original in-memory data was not modified.
if original_data != preprocessing_original_snapshot:
    raise AssertionError(
        'original_data changed during preprocessing.'
    )

# Count invalid timestamp fields.
preprocessing_invalid_timestamp_field_count = sum(
    preprocessing_counts.get('invalid', 0)
    for preprocessing_counts in preprocessing_timestamp_counts.values()
)

# Display the actual results.
print("PREPROCESSING SUMMARY")
print("=" * 45)
print(f'Cleaned JSON: {preprocessing_output_path}')
print(f'Original record count: {preprocessing_original_count:,}')
print(f'Cleaned record count: {len(preprocessing_cleaned_data):,}')
print(f'Duplicate records removed: {preprocessing_duplicate_records_removed}')
print(f'Missing IDs retained: {preprocessing_missing_id_count}')
print(f'Invalid timestamp fields: {preprocessing_invalid_timestamp_field_count}')
print(f'Records with invalid timestamps: {len(preprocessing_invalid_timestamp_records)}')
print(f'Inconsistent resolution durations: {preprocessing_resolution_mismatch_count}')
print(f'Invalid comment counts set to null: {preprocessing_invalid_comments_count}')
print(f'Numeric comment strings converted: {preprocessing_comment_strings_converted}')
print(f'Original in-memory data unchanged: {original_data == preprocessing_original_snapshot}')

print("\nREPRESENTATIVE CLEANED DOCUMENT")
print("=" * 45)

preprocessing_pprint(
    preprocessing_cleaned_data[0] if preprocessing_cleaned_data else None,
    sort_dicts=False,
    width=100
)

print("\nAll final validation checks passed.")


PREPROCESSING SUMMARY
Cleaned JSON: c:\Users\riya choudhary\OneDrive\Desktop\BugRadar\data\bugradar_issues_cleaned.json
Original record count: 5,000
Cleaned record count: 5,000
Duplicate records removed: 0
Missing IDs retained: 0
Invalid timestamp fields: 22
Records with invalid timestamps: 22
Inconsistent resolution durations: 0
Invalid comment counts set to null: 0
Numeric comment strings converted: 130
Original in-memory data unchanged: True

REPRESENTATIVE CLEANED DOCUMENT
{'id': 1000000,
 'number': 1,
 'title': 'Example missing for refresh the page',
 'body': None,
 'issue_category': 'documentation',
 'state': 'open',
 'state_reason': None,
 'priority': 'medium',
 'created_at': '2025-08-05T17:32:00Z',
 'closed_at': None,
 'resolution_hours': None,
 'labels': [],
 'comments': 7,
 'user': {'login': None, 'type': None},
 'assignees': [],
 'repository': {'name': 'auth-api', 'language': 'TypeScript', 'type': 'backend', 'stars': 144},
 'environment': {'platform': 'Firefox 131', 'version